# Ethos Academy Oversight Agent — Evaluation Notebook

Interactively evaluate the oversight agent on the held-out test set.
Works for both the **base model** (pre-GRPO baseline) and a **trained LoRA adapter** (post-GRPO).

## Sections
1. **Setup** — install dependencies, configure paths
2. **Load model** — base model or base + LoRA adapter
3. **Single-record inspection** — see the full `<think>` reasoning + extracted verdict vs. ground truth
4. **Batch metrics** — run over all test records, display metrics table
5. **Side-by-side comparison** — compare base model (no RL) vs. GRPO+LoRA on the same 20 samples *(optional)*

---
**Hardware note:** Llama-3.2-1B-FP8 fits on a free Colab T4 (16 GB).  
For Qwen3-8B-FP8, use Colab Pro with L4 (22 GB).

## 1. Setup

In [ ]:
import os

# Detect Colab and install dependencies
IN_COLAB = 'COLAB_' in ''.join(os.environ.keys())
if IN_COLAB:
    import subprocess
    is_t4 = 'Tesla T4' in str(subprocess.check_output(['nvidia-smi']))
    vllm_ver = 'vllm==0.9.2' if is_t4 else 'vllm==0.15.1'
    triton_ver = 'triton==3.2.0' if is_t4 else 'triton'
    os.system(f'pip install -qqq {vllm_ver} torchvision bitsandbytes xformers unsloth')
    os.system(f'pip install -qqq {triton_ver}')
    os.system('pip install transformers==4.56.2')
    os.system('pip install --no-deps trl==0.22.2')
    # Clone repo if not already present
    if not os.path.exists('DataMassageForGRPO'):
        os.system('git clone https://github.com/YOUR_ORG/DataMassageForGRPO.git')
    os.chdir('DataMassageForGRPO/grpo-pipeline')
    os.system('pip install -e ".[train]" -qqq')
print('Setup complete.')

In [ ]:
# ── Configure paths and model here ──────────────────────────────────────────

# Path to test.jsonl (relative to the grpo-pipeline/ directory)
TEST_FILE = '../transformed/test.jsonl'

# Base model — Phase 1 default (T4 compatible)
MODEL_NAME = 'unsloth/Llama-3.2-1B-Instruct-FP8-Block'
# Phase 2 — uncomment for Qwen3-8B on Colab Pro L4:
# MODEL_NAME = 'unsloth/Qwen3-8B-FP8'

# LoRA adapter path — set to None for base model (baseline), or a path string for post-GRPO
LORA_ADAPTER = None
# LORA_ADAPTER = '../lora-adapter'

MAX_SEQ_LENGTH   = 2048
MAX_NEW_TOKENS   = 768
INFERENCE_BATCH  = 4    # Reduce to 1 or 2 if OOM

print(f'Model:        {MODEL_NAME}')
print(f'LoRA adapter: {LORA_ADAPTER or "(none — baseline)"}')
print(f'Test file:    {TEST_FILE}')

## 2. Load Model

In [ ]:
import torch
from unsloth import FastLanguageModel

print(f'Loading {MODEL_NAME} ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    dtype=None,
)

if LORA_ADAPTER is not None:
    from peft import PeftModel
    print(f'Loading LoRA adapter from {LORA_ADAPTER} ...')
    model = PeftModel.from_pretrained(model, LORA_ADAPTER)

FastLanguageModel.for_inference(model)
model.eval()

mode_label = f'Post-GRPO ({LORA_ADAPTER})' if LORA_ADAPTER else 'Baseline (no adapter)'
print(f'Ready: {mode_label}')

## 3. Single-Record Inspection

Set `RECORD_INDEX` to any integer to inspect that record from the test set.

In [ ]:
import json
from grpo_pipeline.rewards import extract_verdict, extract_think
from grpo_pipeline.transform import SYSTEM_PROMPT_TEMPLATE

# Load all test records
with open(TEST_FILE) as f:
    test_records = [json.loads(line) for line in f if line.strip()]

print(f'Loaded {len(test_records)} test records.')

In [ ]:
RECORD_INDEX = 0  # ← change this to inspect a different record

rec = test_records[RECORD_INDEX]

# Build full prompt
messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT_TEMPLATE.format(author=rec['author'])},
    *rec['prompt'],
]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print('=== PROMPT (last 500 chars) ===')
print(prompt_text[-500:])
print()
print('=== GROUND TRUTH ===')
print(f'  alignment: {rec["ground_truth_alignment"]}')
print(f'  safety_score: {rec["ground_truth_safety_score"]:.3f}')
print(f'  length_scale: {rec["length_scale"]:.3f}  (turn {rec["turn_index"]}/{rec["total_turns"]-1})')
print(f'  traits: {json.dumps(rec["ground_truth_traits"], indent=4)}')

In [ ]:
# Run inference on the selected record
inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True,
                   max_length=MAX_SEQ_LENGTH - MAX_NEW_TOKENS).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

input_len = inputs['input_ids'].shape[1]
generated = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)

think = extract_think(generated)
verdict = extract_verdict(generated)

print('=== THINK ===')
print(think or '(no <think> block found)')
print()
print('=== VERDICT ===')
if verdict:
    print(json.dumps(verdict, indent=2))
    print()
    print('=== COMPARISON ===')
    gt_align = rec['ground_truth_alignment']
    pred_align = verdict.get('alignment_status')
    match = '✓' if pred_align == gt_align else '✗'
    print(f'  alignment: predicted={pred_align}  ground_truth={gt_align}  {match}')
    print()
    print(f'  {"Trait":<15} {"Predicted":>12} {"Ground Truth":>14} {"Error":>8}')
    print(f'  {"-" * 54}')
    for k in rec['ground_truth_traits']:
        pred_v = verdict.get(k, float('nan'))
        gt_v   = rec['ground_truth_traits'][k]
        err    = abs(pred_v - gt_v)
        print(f'  {k:<15} {pred_v:>12.3f} {gt_v:>14.3f} {err:>8.3f}')
else:
    print('(could not parse verdict JSON)')
    print()
    print('Raw output:')
    print(generated)

## 4. Batch Metrics

Runs inference over all test records and computes the full evaluation metrics table.  
This may take a few minutes on T4.

In [ ]:
from tqdm.auto import tqdm
from grpo_pipeline.baseline import compute_metrics, print_metrics

batch_results = []
records_to_eval = test_records  # change to test_records[:50] for a quick smoke test

for i in tqdm(range(0, len(records_to_eval), INFERENCE_BATCH), desc='Evaluating'):
    batch = records_to_eval[i : i + INFERENCE_BATCH]

    prompt_texts = [
        tokenizer.apply_chat_template(
            [{'role': 'system', 'content': SYSTEM_PROMPT_TEMPLATE.format(author=r['author'])},
             *r['prompt']],
            tokenize=False,
            add_generation_prompt=True,
        )
        for r in batch
    ]

    enc = tokenizer(
        prompt_texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH - MAX_NEW_TOKENS,
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    in_len = enc['input_ids'].shape[1]
    for rec, out in zip(batch, out_ids):
        text = tokenizer.decode(out[in_len:], skip_special_tokens=True)
        parsed = extract_verdict(text)
        batch_results.append({
            'gt_alignment': rec['ground_truth_alignment'],
            'gt_traits':    rec['ground_truth_traits'],
            'parsed_verdict': parsed,
        })

print(f'\nInference complete. {len(batch_results)} records processed.')

In [ ]:
metrics = compute_metrics(batch_results)
print_metrics(metrics, label=mode_label)

## 5. Side-by-Side Comparison: Base Model vs. GRPO+LoRA *(optional)*

Load the GRPO-trained LoRA adapter and compare its alignment verdicts against the **base model (no RL training)**
on the same 20 records. Shows where GRPO changed the model's judgments and whether those changes moved
toward ground truth.

Only run this section after you have a trained adapter from `train.ipynb`.

In [ ]:
COMPARE_LORA_PATH = '../lora-adapter'  # ← set to your trained adapter path
N_COMPARE = 20

# Load the LoRA adapter on top of the base model
from peft import PeftModel
model_lora, tokenizer_lora = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    dtype=None,
)
model_lora = PeftModel.from_pretrained(model_lora, COMPARE_LORA_PATH)
FastLanguageModel.for_inference(model_lora)
model_lora.eval()
print('LoRA model ready.')

In [ ]:
def run_single(m, tok, rec):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT_TEMPLATE.format(author=rec['author'])},
            *rec['prompt']]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors='pt',
              truncation=True, max_length=MAX_SEQ_LENGTH - MAX_NEW_TOKENS).to(m.device)
    with torch.no_grad():
        out = m.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)

compare_records = test_records[:N_COMPARE]
rows = []

for rec in tqdm(compare_records, desc='Comparing'):
    base_text = run_single(model, tokenizer, rec)
    lora_text = run_single(model_lora, tokenizer_lora, rec)
    base_v    = extract_verdict(base_text)
    lora_v    = extract_verdict(lora_text)
    rows.append({
        'author':        rec['author'],
        'gt':            rec['ground_truth_alignment'],
        'base':          base_v.get('alignment_status') if base_v else None,
        'lora':          lora_v.get('alignment_status') if lora_v else None,
        'length_scale':  rec['length_scale'],
    })

print(f'\n{"Author":<20} {"Ground Truth":<14} {"Base":<14} {"LoRA":<14} {"Scale":>6}')
print('-' * 72)
for r in rows:
    base_mark = '✓' if r['base'] == r['gt'] else ('✗' if r['base'] else '?')
    lora_mark = '✓' if r['lora'] == r['gt'] else ('✗' if r['lora'] else '?')
    changed   = ' ←changed' if r['base'] != r['lora'] else ''
    print(f"{r['author']:<20} {r['gt']:<14} {(r['base'] or '?')+' '+base_mark:<14}"
          f" {(r['lora'] or '?')+' '+lora_mark:<14} {r['length_scale']:>6.2f}{changed}")